# Test persisted intent ensemble

This notebook verifies that the saved `IntentClassifierEnsemble` can be loaded and used for inference without retraining.

In [1]:
from pathlib import Path

from agent1.classification.ensemble import IntentClassifierEnsemble

In [2]:
# Notebook is expected to live in: agent1/notebooks/
PROJECT_ROOT = Path.cwd().parent

MODEL_PATH = (
    PROJECT_ROOT
    / "models"
    / "intent_classifier"
    / "ensemble.joblib"
)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("MODEL_PATH:   ", MODEL_PATH)
print("Model exists:", MODEL_PATH.exists())

PROJECT_ROOT: d:\Projects\projects\agent1
MODEL_PATH:    d:\Projects\projects\agent1\models\intent_classifier\ensemble.joblib
Model exists: True


In [3]:
ensemble_model = IntentClassifierEnsemble.load(MODEL_PATH)

print("Model fitted:", ensemble_model.is_fitted())
print("Number of intent classes:", len(ensemble_model.classes_))

Model fitted: True
Number of intent classes: 46


In [4]:
test_queries = [
    "Where is my order?",
    "Can you help me find a new pair of running shoes?",
]

predictions = ensemble_model.predict(test_queries)

for query, intent in zip(test_queries, predictions):
    print(f"Query: {query}")
    print(f"Predicted intent: {intent}")
    print()

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Query: Where is my order?
Predicted intent: track_order

Query: Can you help me find a new pair of running shoes?
Predicted intent: store_location



In [5]:
confidence_results = ensemble_model.predict_with_confidence(test_queries)

for query, result in zip(test_queries, confidence_results):
    print(f"Query: {query}")
    print(f"Intent: {result['intent']}")
    print(f"Score: {result['score']:.4f}")
    print(f"Second intent: {result['second_intent']}")
    print(f"Second score: {result['second_score']:.4f}")
    print(f"Margin: {result['margin']:.4f}")
    print()

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Query: Where is my order?
Intent: track_order
Score: 0.6426
Second intent: track_delivery
Second score: 0.0313
Margin: 0.6113

Query: Can you help me find a new pair of running shoes?
Intent: store_location
Score: 0.1801
Second intent: track_order
Second score: 0.1171
Margin: 0.0631



In [6]:
top_k_results = ensemble_model.predict_top_k(test_queries, k=3)

for query, results in zip(test_queries, top_k_results):
    print(f"Query: {query}")
    for rank, item in enumerate(results, start=1):
        print(f"  {rank}. {item['intent']} ({item['score']:.4f})")
    print()

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Query: Where is my order?
  1. track_order (0.6426)
  2. track_delivery (0.0313)
  3. order_history (0.0269)

Query: Can you help me find a new pair of running shoes?
  1. store_location (0.1801)
  2. track_order (0.1171)
  3. request_refund (0.0605)

